# Finetuning Qwen2.5-VL-3B for PKR Currency Notes + Fruits/Vegetables (v3: fast)

LoRA-finetunes **Qwen2.5-VL-3B-Instruct** (via [Unsloth](https://github.com/unslothai/unsloth)) so it answers "what is this?" with a short, TTS-friendly answer like `"1000 rupee note"` or `"banana"`.


**Runtime:** Colab, `Runtime > Change runtime type > A100 GPU`.

> For a closed set of ~8 notes + ~27 fruits/veg, a small CNN classifier (MobileNetV3 / EfficientNet-B0) is still much cheaper to train and serve than a VLM. This notebook keeps the VLM path since that's what you asked for.

## 1. Install dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U "trl>=0.15.0" peft accelerate bitsandbytes triton kagglehub


## 2. Kaggle API access

Create a `kaggle.json` file (`{"username":"YOUR_USERNAME","key":"YOUR_KEY"}`), then run the cell below and upload it.

In [ ]:
import os
from google.colab import files

if not os.path.exists("/root/.kaggle/kaggle.json"):
    uploaded = files.upload()  # choose your kaggle.json file
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fn in uploaded:
        os.rename(fn, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

import kagglehub
print("Kaggle credentials set.")


## 3. Config + helpers (image resizing is the key speed fix)

In [ ]:
import os, gc, json, time, random, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps

CURRENCY_DIR = "/content/currency_dataset"   # resized copies are written here
EDIBLES_DIR  = "/content/edibles_dataset"

IMAGE_INSTRUCTION = "What is shown in this image? Answer with just the item name."

# ---------------- speed settings ----------------
MAX_SIDE               = 448   # longest image side after resizing (=> ~100-250 visual tokens)
CURRENCY_MAX_PER_CLASS = 300   # per currency class
EDIBLES_MAX_PER_LABEL  = 100   # per fruit/veg label (pooled across all Fruits-360 folders of that fruit)
TIME_BUDGET_MIN        = 90    # training stops itself after this many minutes
# ------------------------------------------------

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")

def print_tree(root, max_depth=3, max_items=15):
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(dirpath) or dirpath}/")
        for f in sorted(filenames)[:3]:
            print(f"{indent}  {f}")
        if len(filenames) > 3:
            print(f"{indent}  ... ({len(filenames)} files total)")

def resize_and_save(job):
    """Decode -> fix phone-camera rotation -> shrink to MAX_SIDE -> save as JPEG."""
    src, dst = job
    try:
        with Image.open(src) as im:
            if im.format == "JPEG":
                im.draft("RGB", (MAX_SIDE * 2, MAX_SIDE * 2))   # very fast downscale-on-decode for big JPEGs
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
            im.save(dst, "JPEG", quality=92)
        return True
    except Exception as e:
        print("skipped", src, "->", e)
        return False

def build_resized_dataset(files_by_label, dst_root, max_per_label, seed=3407):
    """files_by_label: {label: [source image paths]}  ->  resized JPEGs in dst_root/<label>/ (capped per label)."""
    shutil.rmtree(dst_root, ignore_errors=True)
    rng = random.Random(seed)
    jobs = []
    for label, files in files_by_label.items():
        files = sorted(files)
        rng.shuffle(files)
        out_dir = os.path.join(dst_root, label)
        os.makedirs(out_dir, exist_ok=True)
        for i, src in enumerate(files[:max_per_label]):
            jobs.append((src, os.path.join(out_dir, f"{i:05d}.jpg")))
    with ThreadPoolExecutor(max_workers=8) as ex:
        ok = list(ex.map(resize_and_save, jobs))
    print(f"Resized {sum(ok)}/{len(jobs)} images -> {dst_root}")

## 4. Download + resize the PKR currency dataset (Kaggle)

Every folder that directly contains images is treated as one class. The cell prints a few **original** image sizes so you can see how big the raw photos were.

In [ ]:
import kagglehub
currency_raw_path = kagglehub.dataset_download("muhammadammarabid/pakistanicurrencydataset")
print("Downloaded to:", currency_raw_path)
print_tree(currency_raw_path, max_depth=5)

In [ ]:
def find_images_by_class(src_root):
    files_by_class = {}
    for dirpath, _, filenames in os.walk(src_root):
        imgs = [os.path.join(dirpath, f) for f in filenames if f.lower().endswith(IMG_EXT)]
        if imgs:
            files_by_class.setdefault(os.path.basename(dirpath), []).extend(imgs)
    return files_by_class

currency_files = find_images_by_class(currency_raw_path)
print("Images per class (original):", {k: len(v) for k, v in sorted(currency_files.items())})

print("\nOriginal image sizes (width, height) -- big ones are what made training so slow:")
for cls, paths in list(sorted(currency_files.items()))[:5]:
    with Image.open(paths[0]) as im:
        print(f"  {cls:25} {im.size}")

build_resized_dataset(currency_files, CURRENCY_DIR, CURRENCY_MAX_PER_CLASS)
print("Currency classes:", sorted(os.listdir(CURRENCY_DIR)))
gc.collect()

## 5. Download Fruits-360 (Kaggle)

In [ ]:
fruits360_path = kagglehub.dataset_download("moltean/fruits")
print("Downloaded to:", fruits360_path)
print_tree(fruits360_path, max_depth=4)


In [ ]:
def find_split_root(root, split_name="Training"):
    for dirpath, dirnames, _ in os.walk(root):
        if split_name in dirnames or split_name.lower() in [d.lower() for d in dirnames]:
            return dirpath
    return None

split_root = find_split_root(fruits360_path)
if split_root is None:
    raise RuntimeError("Couldn't locate a Training/ folder -- check the tree printed above.")

def resolve_split_dir(root, name):
    for d in os.listdir(root):
        if d.lower() == name.lower():
            return os.path.join(root, d)
    return None

train_dir = resolve_split_dir(split_root, "Training")
test_dir  = resolve_split_dir(split_root, "Test")
print("Total available classes:", len(os.listdir(train_dir)))


## 6. Build the fruit/veg set -- pooled per fruit, capped, resized

`EDIBLE_KEYWORDS` picks the fruits/vegetables to keep (edit freely). Matching is by the **first word of the folder name**, so `"Apple Red 1"`, `"Apple Golden 3"`, ... all pool into one `apple` class, and then **100 images per fruit** are sampled from that pool.

In [ ]:
EDIBLE_KEYWORDS = [
    "apple", "banana", "orange", "mango", "grape", "pear", "pineapple",
    "watermelon", "kiwi", "lemon", "pomegranate", "strawberry", "peach", "plum",
    "potato", "tomato", "onion", "carrot", "cucumber", "pepper", "cabbage",
    "cauliflower", "corn", "garlic", "ginger", "eggplant", "beetroot",
]

def label_to_answer(folder_name: str, category: str) -> str:
    """Turn a folder name into a short spoken-friendly answer."""
    name = folder_name.strip().lower().replace("_", " ").replace("-", " ")
    if category == "currency":
        digits = "".join(ch for ch in name if ch.isdigit())
        if digits:
            return f"{digits} rupee note"
        return f"{name} note"
    else:
        return name.split()[0]

edible_files = {}
for split_dir in (train_dir, test_dir):
    if split_dir is None:
        continue
    for cls in os.listdir(split_dir):
        label = label_to_answer(cls, "edible")
        cls_src = os.path.join(split_dir, cls)
        if label not in EDIBLE_KEYWORDS or not os.path.isdir(cls_src):
            continue
        edible_files.setdefault(label, []).extend(
            os.path.join(cls_src, f) for f in os.listdir(cls_src) if f.lower().endswith(IMG_EXT)
        )

print("Available images per kept fruit/veg:", {k: len(v) for k, v in sorted(edible_files.items())})
missing = [k for k in EDIBLE_KEYWORDS if k not in edible_files]
if missing:
    print("Keywords with no match in Fruits-360 (ignored):", missing)

build_resized_dataset(edible_files, EDIBLES_DIR, EDIBLES_MAX_PER_LABEL)
gc.collect()

In [ ]:
def collect_examples(root_dir, category):
    examples = []
    root = Path(root_dir)
    if not root.exists():
        print(f"WARNING: {root_dir} does not exist, skipping.")
        return examples
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        answer = label_to_answer(class_dir.name, category)
        for img_path in sorted(class_dir.glob("*.jpg")):
            examples.append({"image_path": str(img_path), "answer": answer, "category": category})
    return examples

currency_examples = collect_examples(CURRENCY_DIR, "currency")
edible_examples   = collect_examples(EDIBLES_DIR, "edible")
print(f"Currency images: {len(currency_examples)}")
print(f"Edible images:   {len(edible_examples)}")

all_examples = currency_examples + edible_examples
random.seed(3407)
random.shuffle(all_examples)

split_idx = int(0.9 * len(all_examples))
train_examples = all_examples[:split_idx]
val_examples   = all_examples[split_idx:]
print(f"Train: {len(train_examples)}  Val: {len(val_examples)}")
print(f"Steps per epoch at batch size 16: ~{len(train_examples)//16}")

# Optional backup of the dataset list (lost only if the Colab session is fully disconnected)
with open("/content/dataset_checkpoint.json", "w") as f:
    json.dump({"train": train_examples, "val": val_examples}, f)

## 7. Load Qwen2.5-VL-3B-Instruct in 16-bit with Unsloth

16-bit (bf16) instead of 4-bit: a 3B model is only ~6 GB, and it trains faster on an A100 without the 4-bit dequantization overhead.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-3B-Instruct",
    load_in_4bit = False,
    use_gradient_checkpointing = "unsloth",
)

## 8. Add LoRA adapters

Vision layers stay frozen (the pretrained visual features are already good enough to tell a banana from a tomato). Flip `finetune_vision_layers` to `True` only if the accuracy check in step 12 says it's needed.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # off by default for speed -- see note above
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 9. Convert to Unsloth's vision conversation format (lazy) + token-count check

In [ ]:
from datasets import Dataset as HFDataset

def convert_to_conversation(example):
    image = Image.open(example["image_path"]).convert("RGB")
    image.thumbnail((MAX_SIDE, MAX_SIDE))   # safety net; images are already resized on disk
    conversation = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": IMAGE_INSTRUCTION},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": example["answer"]},
        ]},
    ]
    return {"messages": conversation}

def apply_conversation_transform(batch):
    return {
        "messages": [
            convert_to_conversation({"image_path": p, "answer": a})["messages"]
            for p, a in zip(batch["image_path"], batch["answer"])
        ]
    }

VAL_DURING_TRAINING = 200   # small val subset for the periodic eval (full val set is used in step 12)

converted_train_dataset = HFDataset.from_list(train_examples)
converted_val_dataset   = HFDataset.from_list(val_examples[:VAL_DURING_TRAINING])
converted_train_dataset.set_transform(apply_conversation_transform)
converted_val_dataset.set_transform(apply_conversation_transform)
print("Train:", len(converted_train_dataset), " Val (during training):", len(converted_val_dataset))

In [ ]:
# Sanity check: how many tokens does one sample really cost? (should be roughly 100-300, not thousands)
for ex in train_examples[:3]:
    img = Image.open(ex["image_path"]).convert("RGB")
    msgs = [{"role": "user", "content": [{"type": "image", "image": img},
                                         {"type": "text", "text": IMAGE_INSTRUCTION}]}]
    txt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inp = tokenizer(img, txt, add_special_tokens=False, return_tensors="pt")
    print(f"image {img.size} -> {inp['input_ids'].shape[1]} tokens   ({ex['answer']})")

## 10. Train

- Batch 16, `dataloader_num_workers = 8`, `max_seq_length = 512`
- Eval every 50 steps on 200 val images; checkpoint every 50 steps to `outputs_fast/`; best checkpoint is restored at the end
- Early stopping (patience 4) **and** a hard time limit (`TIME_BUDGET_MIN`)
- If the runtime crashes but `/content` survives, just re-run this cell: it resumes from the latest checkpoint.

Watch the progress bar for the first minute. If the ETA is well above your time budget, stop and lower `MAX_SIDE` (e.g. 336) or `EDIBLES_MAX_PER_LABEL` in step 3 and re-run from there.

In [ ]:
import glob
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback, TrainerCallback

class TimeBudgetCallback(TrainerCallback):
    """Stops training (and saves) once the wall-clock budget is used up."""
    def __init__(self, max_minutes):
        self.max_seconds = max_minutes * 60
        self.start = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.start = time.time()
    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.start > self.max_seconds:
            print(f"\nTime budget of {self.max_seconds/60:.0f} min reached -- stopping and saving.")
            control.should_save = True
            control.should_training_stop = True
        return control

gc.collect()
torch.cuda.empty_cache()
FastVisionModel.for_training(model)

OUTPUT_DIR = "outputs_fast"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_train_dataset,
    eval_dataset = converted_val_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 16,
        per_device_eval_batch_size = 16,
        gradient_accumulation_steps = 1,   # effective batch size 16
        dataloader_num_workers = 8,
        dataloader_pin_memory = True,
        warmup_steps = 10,
        num_train_epochs = 2,              # upper bound; early stopping / time budget may end it sooner
        learning_rate = 2e-4,
        max_grad_norm = 0.3,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 50,
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = OUTPUT_DIR,
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = 2,
        max_seq_length = 512,
    ),
    callbacks = [
        EarlyStoppingCallback(early_stopping_patience = 4),
        TimeBudgetCallback(TIME_BUDGET_MIN),
    ],
)

resume = len(glob.glob(f"{OUTPUT_DIR}/checkpoint-*")) > 0
t0 = time.time()
trainer_stats = trainer.train(resume_from_checkpoint = resume)
print(f"\nTraining finished in {(time.time() - t0)/60:.1f} minutes")

## 11. Quick spot check

In [ ]:
FastVisionModel.for_inference(model)

def ask(image_path, instruction=IMAGE_INSTRUCTION):
    image = Image.open(image_path)
    image = ImageOps.exif_transpose(image).convert("RGB")
    image.thumbnail((MAX_SIDE, MAX_SIDE))          # ALWAYS resize like in training (also in your app!)
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": instruction},
    ]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=16, use_cache=True, do_sample=False)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

for ex in val_examples[:5]:
    pred = ask(ex["image_path"])
    print(f"true={ex['answer']!r:25} pred={pred!r:25} path={ex['image_path']}")

## 12. Accuracy on the validation set

Measured on up to `EVAL_N` random held-out images (a loose "contains" match, so `"a banana"` counts as `banana`).

One caveat: Fruits-360 photos are frames of the same fruit rotating on a turntable, so near-duplicates can land in both train and val and make the **fruit** accuracy look better than it will be on real camera photos. The currency number is more trustworthy. Test with a few real photos from your phone before relying on either.

In [ ]:
EVAL_N = 300
random.seed(0)
eval_subset = random.sample(val_examples, min(EVAL_N, len(val_examples)))

t0 = time.time()
correct = 0
results = []
for ex in eval_subset:
    pred = ask(ex["image_path"]).strip().lower()
    is_correct = ex["answer"].lower() in pred
    results.append((ex["answer"], pred, is_correct))
    correct += int(is_correct)

accuracy = correct / len(eval_subset) if eval_subset else 0
print(f"Validation accuracy: {accuracy:.1%}  ({correct}/{len(eval_subset)})   [{time.time()-t0:.0f}s]")

from collections import defaultdict
by_cat = defaultdict(lambda: [0, 0])
for ex, (true, pred, ok) in zip(eval_subset, results):
    by_cat[ex["category"]][0] += int(ok)
    by_cat[ex["category"]][1] += 1
for cat, (c, t) in by_cat.items():
    print(f"  {cat}: {c}/{t} = {c/t:.1%}")

print("\nSample mistakes:")
for true, pred, ok in [r for r in results if not r[2]][:20]:
    print(f"  true={true!r}  pred={pred!r}")

## 13. Save the finetuned model -- download it

`/content` disappears when the session ends, so this zips the LoRA adapter and triggers a browser download. **Do this before your session ends.**

In [ ]:
LORA_SAVE_DIR = "lora_model_your_eyes"
model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)

import shutil
zip_path = shutil.make_archive(LORA_SAVE_DIR, "zip", LORA_SAVE_DIR)
print("Zipped to", zip_path)

from google.colab import files
files.download(zip_path)

# Optional: merge LoRA into the base model and push to Hugging Face Hub for easy serving instead
# model.push_to_hub_merged("your-username/your-eyes-qwen2.5-vl", tokenizer, save_method = "merged_16bit", token = "hf_...")


## Deployment note

- **Resize every image to `MAX_SIDE = 448` (longest side) before sending it to the model**, exactly like `ask()` does. The model was trained on small images; sending full camera frames would be both slow and a mismatch. In your Flutter app, downscale before upload.
- Qwen2.5-VL-3B needs a GPU to serve at usable latency (e.g. vLLM on a small cloud GPU). It won't run inside a Firebase Cloud Function.